# Q-factorisation on Four Rooms

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import random
import copy
from dataclasses import dataclass

# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from environments.fourrooms_discrete import FourRoomsGridWorld, FourRoomsGoalWrapper
from utils import TrajectoryReplayBufferDiscrete, evaluate_policy
from visualisations import plot_policy_rollouts, plot_q_diagnostics
from networks import DQN_QNetwork


DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
class FactorisedDQN_QNetwork(nn.Module):

    def __init__(
        self,
        obs_dim: int,
        num_actions: int,
        goal_dim: int = 2,
        hidden_dim: int = 128,
        rep_dim: int = 64,
    ):
        super().__init__()
        self.num_actions = num_actions
        self.goal_dim = goal_dim
        self.rep_dim = rep_dim

        # Environment / state encoder: s -> phi_s(s) in R^rep_dim
        self.obs_encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, rep_dim),
            nn.ReLU(),
        )

        # Action embedding: a -> e_a in R^rep_dim
        self.action_emb = nn.Embedding(num_actions, rep_dim)

        # Goal / task encoder: z -> psi(z) in R^rep_dim
        self.goal_encoder = nn.Sequential(
            nn.Linear(goal_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, rep_dim),
        )

    def forward(self, obs: torch.Tensor, goal: torch.Tensor) -> torch.Tensor:
        B = obs.shape[0]

        phi_s = torch.tanh(self.obs_encoder(obs))
        psi_z = torch.tanh(self.goal_encoder(goal))

        phi_s = F.normalize(phi_s, p=2, dim=-1, eps=1e-8)
        psi_z = F.normalize(psi_z, p=2, dim=-1, eps=1e-8)
        action_emb = F.normalize(torch.tanh(self.action_emb.weight), p=2, dim=-1, eps=1e-8)
        phi_sa = F.normalize(phi_s.unsqueeze(1) * action_emb.unsqueeze(0), p=2, dim=-1, eps=1e-8)
        q_vals = (phi_sa * psi_z.unsqueeze(1)).sum(dim=-1)
        
        return q_vals

In [ ]:
def make_env(goal=(9, 9), slip_prob=0.00, max_horizon=500):
    base = FourRoomsGridWorld(room_size=5, max_episode_steps=max_horizon)
    env = FourRoomsGoalWrapper(
        base,
        goal_position=goal,
        goal_reward=1.0,
        step_reward=0.0,
        slip_prob=slip_prob,
    )
    return env


In [ ]:
BUFFER_CAPACITY = 100000
GOAL = (9, 9)
LR = float(1e-3)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_goal_tensor(goal, device=DEVICE):
    goal_arr = np.array(goal, dtype=np.float32)
    return torch.tensor(goal_arr, dtype=torch.float32, device=device).unsqueeze(0)


def make_factorised_policy_fn(q_network, goal, device=DEVICE):
    goal_t = make_goal_tensor(goal, device=device)

    def policy_fn(obs):
        obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            q_vals = q_network(obs_t, goal_t)
        return int(q_vals.argmax(dim=-1).item())

    return policy_fn


def make_factorised_value_fn(q_network, goal, device=DEVICE):
    goal_arr = np.array(goal, dtype=np.float32)

    def value_fn(obs_batch):
        obs_t = torch.tensor(obs_batch, dtype=torch.float32, device=device)
        goal_t = torch.tensor(goal_arr, dtype=torch.float32, device=device).unsqueeze(0)
        goal_batch = goal_t.expand(obs_t.shape[0], -1)
        with torch.no_grad():
            q_vals = q_network(obs_t, goal_batch).cpu().numpy()
        return q_vals

    return value_fn


def evaluate_goal_conditioned_policy(
    q_network,
    goal,
    episodes=32,
    seed=0,
    slip_prob=0.0,
    max_horizon=500,
    device=DEVICE,
):
    set_seed(seed)
    eval_env = make_env(goal=goal, slip_prob=slip_prob, max_horizon=max_horizon)
    policy_fn = make_factorised_policy_fn(q_network, goal, device=device)
    mean_ret, mean_len = evaluate_policy(eval_env, policy_fn, episodes=episodes)
    eval_env.close()
    success_rate = float(mean_ret)  # valid here because goal_reward=1, step_reward=0 [file:133]
    return {
        "success_rate": success_rate,
        "mean_return": float(mean_ret),
        "mean_len": float(mean_len),
    }


def set_trainable_modules(model, train_obs_encoder, train_action_emb, train_goal_encoder):
    for p in model.obs_encoder.parameters():
        p.requires_grad_(train_obs_encoder)
    for p in model.action_emb.parameters():
        p.requires_grad_(train_action_emb)
    for p in model.goal_encoder.parameters():
        p.requires_grad_(train_goal_encoder)


def get_trainable_params(model):
    return [p for p in model.parameters() if p.requires_grad]


@dataclass
class TransferConfig:
    name: str
    train_obs_encoder: bool
    train_action_emb: bool
    train_goal_encoder: bool
    init_from_source: bool = True


def train_factorised_on_goal(
    q_network,
    q_target_network,
    env,
    goal,
    *,
    buffer_capacity=100000,
    lr_all=1e-3,
    lr_goal=3e-3,
    device=DEVICE,
    total_steps=30000,
    warmup_steps=5000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_steps=20000,
    train_freq=4,
    eval_every=1000,
    eval_episodes=32,
    seed=0,
):
    set_seed(seed)

    obs_dim = env.observation_space.shape[0]
    buffer = TrajectoryReplayBufferDiscrete(buffer_capacity, obs_dim, 1, device=device)

    trainable_named_params = []
    if any(p.requires_grad for p in q_network.obs_encoder.parameters()):
        trainable_named_params.append({"params": q_network.obs_encoder.parameters(), "lr": lr_all})
    if any(p.requires_grad for p in q_network.action_emb.parameters()):
        trainable_named_params.append({"params": q_network.action_emb.parameters(), "lr": lr_all})
    if any(p.requires_grad for p in q_network.goal_encoder.parameters()):
        trainable_named_params.append({"params": q_network.goal_encoder.parameters(), "lr": lr_goal})

    opt = optim.Adam(trainable_named_params)

    goal_t_single = make_goal_tensor(goal, device=device)

    obs, _ = env.reset(seed=seed)
    global_step = 0
    logs = []

    ep_obs, ep_actions, ep_rewards, ep_next_obs, ep_terminated, ep_truncated = [], [], [], [], [], []

    while global_step < total_steps:
        frac = min(1.0, global_step / eps_decay_steps)
        eps = eps_start + frac * (eps_end - eps_start)

        if np.random.random() < eps:
            action = env.action_space.sample()
        else:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                q_vals = q_network(obs_t, goal_t_single)
                action = int(q_vals.argmax(dim=-1).item())

        next_obs, rew, term, trunc, _ = env.step(action)
        done = term or trunc

        ep_obs.append(obs.copy())
        ep_actions.append(action)
        ep_rewards.append(float(rew))
        ep_next_obs.append(next_obs.copy())
        ep_terminated.append(float(term))
        ep_truncated.append(float(trunc))

        obs = next_obs
        global_step += 1

        if done:
            episode = {
                "obs": ep_obs,
                "actions": ep_actions,
                "rewards": ep_rewards,
                "next_obs": ep_next_obs,
                "terminated": ep_terminated,
                "truncated": ep_truncated,
            }
            buffer.add_episode(episode)
            ep_obs, ep_actions, ep_rewards, ep_next_obs, ep_terminated, ep_truncated = [], [], [], [], [], []
            obs, _ = env.reset()

        if len(buffer) >= warmup_steps and global_step % train_freq == 0:
            batch = buffer.sample(batch_size)

            obs_t = batch.obs
            act_t = batch.actions.long()
            rew_t = batch.rewards
            next_obs_t = batch.next_obs
            term_t = batch.terminated

            goal_batch = goal_t_single.expand(obs_t.shape[0], -1)

            with torch.no_grad():
                next_q_vals = q_target_network(next_obs_t, goal_batch)
                next_q = next_q_vals.max(dim=-1, keepdim=True).values
                target = rew_t + gamma * (1.0 - term_t) * next_q

            current_q_all = q_network(obs_t, goal_batch)
            current_q = current_q_all.gather(1, act_t.unsqueeze(1))

            loss = F.mse_loss(current_q, target)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(get_trainable_params(q_network), 10.0)
            opt.step()

            for p, p_tgt in zip(q_network.parameters(), q_target_network.parameters()):
                p_tgt.data.mul_(1.0 - tau).add_(tau * p.data)

        if global_step % eval_every == 0:
            metrics = evaluate_goal_conditioned_policy(
                q_network=q_network,
                goal=goal,
                episodes=eval_episodes,
                seed=seed + 1000 + global_step,
                device=device,
            )
            logs.append({
                "step": global_step,
                "eps": eps,
                "success_rate": metrics["success_rate"],
                "mean_return": metrics["mean_return"],
                "mean_len": metrics["mean_len"],
            })
            print(
                f"[train={goal}] step={global_step:6d} "
                f"| eps={eps:.3f} "
                f"| success={metrics['success_rate']:.3f} "
                f"| len={metrics['mean_len']:.1f}"
            )

    env.close()
    return q_network, logs

## Single first task training

In [ ]:
# =========================
# Source training
# =========================

SOURCE_GOAL = (9, 9)
SOURCE_SEED = 0

set_seed(SOURCE_SEED)

source_env = make_env(goal=SOURCE_GOAL)
obs_dim = source_env.observation_space.shape[0]
num_actions = source_env.action_space.n

source_q = FactorisedDQN_QNetwork(
    obs_dim=obs_dim,
    num_actions=num_actions,
    goal_dim=2,
    hidden_dim=128,
    rep_dim=64,
).to(DEVICE)

source_q_target = FactorisedDQN_QNetwork(
    obs_dim=obs_dim,
    num_actions=num_actions,
    goal_dim=2,
    hidden_dim=128,
    rep_dim=64,
).to(DEVICE)

source_q_target.load_state_dict(source_q.state_dict())
for p in source_q_target.parameters():
    p.requires_grad_(False)

source_q, source_logs = train_factorised_on_goal(
    q_network=source_q,
    q_target_network=source_q_target,
    env=source_env,
    goal=SOURCE_GOAL,
    total_steps=100000,
    warmup_steps=5000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_steps=50000,
    train_freq=4,
    eval_every=1000,
    eval_episodes=32,
    seed=SOURCE_SEED,
)

source_checkpoint = copy.deepcopy(source_q.state_dict())

# =========================
# Plot source training
# =========================

if len(source_logs) > 0:
    xs = [x["step"] for x in source_logs]
    ys = [x["success_rate"] for x in source_logs]

    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys, label=f"source goal {SOURCE_GOAL}")
    plt.xlabel("Environment steps")
    plt.ylabel("Success rate")
    plt.title("Source training performance")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()

## Now Transfer

In [ ]:
# =========================
# Systematic transfer benchmark
# =========================

TARGET_GOALS = [
    (9, 1),
    (1, 9),
    (1, 1),
    (7, 9),
    (9, 7),
]

TRANSFER_SEEDS = [0, 1, 2]
TRANSFER_TOTAL_STEPS = 30000
TRANSFER_EVAL_EVERY = 1000
TRANSFER_EVAL_EPISODES = 32

transfer_configs = [
    TransferConfig(
        name="psi_only",
        train_obs_encoder=False,
        train_action_emb=False,
        train_goal_encoder=True,
        init_from_source=True,
    ),
    TransferConfig(
        name="full_finetune",
        train_obs_encoder=True,
        train_action_emb=True,
        train_goal_encoder=True,
        init_from_source=True,
    ),
    TransferConfig(
        name="scratch",
        train_obs_encoder=True,
        train_action_emb=True,
        train_goal_encoder=True,
        init_from_source=False,
    ),
]

transfer_results = []

for target_goal in TARGET_GOALS:
    for seed in TRANSFER_SEEDS:
        print(f"\n===== TARGET GOAL {target_goal} | seed={seed} =====")

        # zero-shot from source model
        zero_shot_model = FactorisedDQN_QNetwork(
            obs_dim=obs_dim,
            num_actions=num_actions,
            goal_dim=2,
            hidden_dim=128,
            rep_dim=64,
        ).to(DEVICE)
        zero_shot_model.load_state_dict(source_checkpoint)

        zero_metrics = evaluate_goal_conditioned_policy(
            q_network=zero_shot_model,
            goal=target_goal,
            episodes=TRANSFER_EVAL_EPISODES,
            seed=seed,
            device=DEVICE,
        )

        transfer_results.append({
            "config": "zero_shot",
            "target_goal": target_goal,
            "seed": seed,
            "step": 0,
            "success_rate": zero_metrics["success_rate"],
            "mean_return": zero_metrics["mean_return"],
            "mean_len": zero_metrics["mean_len"],
        })

        print(
            f"[zero_shot] goal={target_goal} seed={seed} "
            f"| success={zero_metrics['success_rate']:.3f} "
            f"| len={zero_metrics['mean_len']:.1f}"
        )

        for cfg in transfer_configs:
            print(f"\n--- transfer config: {cfg.name} | goal={target_goal} | seed={seed} ---")

            model = FactorisedDQN_QNetwork(
                obs_dim=obs_dim,
                num_actions=num_actions,
                goal_dim=2,
                hidden_dim=128,
                rep_dim=64,
            ).to(DEVICE)

            target_model = FactorisedDQN_QNetwork(
                obs_dim=obs_dim,
                num_actions=num_actions,
                goal_dim=2,
                hidden_dim=128,
                rep_dim=64,
            ).to(DEVICE)

            if cfg.init_from_source:
                model.load_state_dict(source_checkpoint)

            target_model.load_state_dict(model.state_dict())
            for p in target_model.parameters():
                p.requires_grad_(False)

            set_trainable_modules(
                model,
                train_obs_encoder=cfg.train_obs_encoder,
                train_action_emb=cfg.train_action_emb,
                train_goal_encoder=cfg.train_goal_encoder,
            )

            env = make_env(goal=target_goal)

            model, logs = train_factorised_on_goal(
                q_network=model,
                q_target_network=target_model,
                env=env,
                goal=target_goal,
                total_steps=TRANSFER_TOTAL_STEPS,
                warmup_steps=5000,
                batch_size=256,
                gamma=0.99,
                tau=0.005,
                eps_start=1.0,
                eps_end=0.05,
                eps_decay_steps=20000,
                train_freq=4,
                eval_every=TRANSFER_EVAL_EVERY,
                eval_episodes=TRANSFER_EVAL_EPISODES,
                seed=seed,
            )

            for row in logs:
                transfer_results.append({
                    "config": cfg.name,
                    "target_goal": target_goal,
                    "seed": seed,
                    "step": row["step"],
                    "success_rate": row["success_rate"],
                    "mean_return": row["mean_return"],
                    "mean_len": row["mean_len"],
                })

In [ ]:
# =========================
# Aggregate transfer curves
# =========================

from collections import defaultdict

grouped = defaultdict(list)
for row in transfer_results:
    key = (row["config"], row["step"])
    grouped[key].append(row["success_rate"])

agg_rows = []
for (config, step), vals in grouped.items():
    agg_rows.append({
        "config": config,
        "step": step,
        "mean_success": float(np.mean(vals)),
        "std_success": float(np.std(vals)),
        "n": len(vals),
    })

configs_in_plot = ["zero_shot", "psi_only", "full_finetune", "scratch"]

plt.figure(figsize=(8, 5))
for cfg_name in configs_in_plot:
    rows = sorted(
        [r for r in agg_rows if r["config"] == cfg_name],
        key=lambda x: x["step"]
    )
    if len(rows) == 0:
        continue
    xs = [r["step"] for r in rows]
    ys = [r["mean_success"] for r in rows]
    plt.plot(xs, ys, label=cfg_name)

plt.xlabel("Adaptation steps")
plt.ylabel("Mean success rate")
plt.title("Transfer performance across goals and seeds")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

# =========================
# Summary table
# =========================

def mean_metric(rows, config, step):
    vals = [r["success_rate"] for r in rows if r["config"] == config and r["step"] == step]
    return float(np.mean(vals)) if len(vals) > 0 else np.nan

all_steps = sorted(set(r["step"] for r in transfer_results))
final_step = max(all_steps)
mid_step = 5000 if 5000 in all_steps else all_steps[min(len(all_steps)-1, 1)]

summary_configs = ["zero_shot", "psi_only", "full_finetune", "scratch"]

print("\n=== TRANSFER SUMMARY ===")
print(f"{'config':<15} {'step0':>10} {'step5k':>10} {'final':>10}")
for cfg in summary_configs:
    s0 = mean_metric(transfer_results, cfg, 0)
    s5 = mean_metric(transfer_results, cfg, mid_step)
    sf = mean_metric(transfer_results, cfg, final_step)
    print(f"{cfg:<15} {s0:10.3f} {s5:10.3f} {sf:10.3f}")